<a href="https://colab.research.google.com/github/tpedCode/P07/blob/feat%2Fapi/notebooks/2_api_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==================================================
# IMPORTS
# ==================================================

import os
import joblib
import pandas as pd

In [4]:
# ==================================================
# MONTAGE DE GOOGLE DRIVE
# ==================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# ==================================================
# CHEMIN DU PROJET
# ==================================================

PROJECT_PATH = (
    "/content/drive/MyDrive/Pro/"
    "Data_Scientist_by_Openclassrooms_en_cours/P07"
)

MODELS_PATH = os.path.join(PROJECT_PATH, "models")

In [6]:
# ==================================================
# VERIFICATION DES FICHIERS
# ==================================================

print(os.listdir(MODELS_PATH))

['threshold.pkl', 'feature_names.pkl', 'best_model.pkl']


In [7]:
# ==================================================
# CHARGEMENT DES ARTEFACTS DU MODELE
# ==================================================

# Modèle retenu lors de la phase de modélisation
model = joblib.load(
    os.path.join(MODELS_PATH, "best_model.pkl")
)

# Liste des variables utilisées pendant l'entraînement
feature_names = joblib.load(
    os.path.join(MODELS_PATH, "feature_names.pkl")
)

# Seuil métier optimal
threshold = joblib.load(
    os.path.join(MODELS_PATH, "threshold.pkl")
)

print("Type de modèle :", type(model))
print("Nombre de variables :", len(feature_names))
print("Seuil métier :", threshold)

Type de modèle : <class 'lightgbm.sklearn.LGBMClassifier'>
Nombre de variables : 246
Seuil métier : 0.09090909090909091


In [8]:
# ==================================================
# APERCU DES VARIABLES D'ENTRAINEMENT
# ==================================================

feature_names[:20]

['CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY']

In [9]:
# ==================================================
# CLIENT DE TEST
# ==================================================

client_data = {
    "AMT_INCOME_TOTAL": 200000,
    "AMT_CREDIT": 300000,
    "AMT_ANNUITY": 15000,
    "EXT_SOURCE_MEAN": 0.60,
    "PAYMENT_RATE": 0.05
}

In [10]:
# ==================================================
# PREPARATION DES DONNEES
# ==================================================

# Création d'un DataFrame à partir du dictionnaire
X = pd.DataFrame([client_data])

# Reconstruction des colonnes attendues
# par le modèle d'entraînement
X = X.reindex(
    columns=feature_names,
    fill_value=0
)

print("Shape :", X.shape)

Shape : (1, 246)


In [11]:
# ==================================================
# CALCUL DE LA PROBABILITE DE DEFAUT
# ==================================================

default_probability = float(
    model.predict_proba(X)[0, 1]
)

print(
    "Probabilité de défaut :",
    round(default_probability, 6)
)

Probabilité de défaut : 0.450369


In [12]:
# ==================================================
# APPLICATION DU SEUIL METIER
# ==================================================

decision = (
    "REFUSED"
    if default_probability >= threshold
    else "ACCEPTED"
)

print("Probabilité :", round(default_probability, 6))
print("Seuil       :", round(float(threshold), 6))
print("Décision    :", decision)

Probabilité : 0.450369
Seuil       : 0.090909
Décision    : REFUSED


In [13]:
# ==================================================
# FONCTION DE PREDICTION
# ==================================================

def predict_client(client_data):
    """
    Calcule la probabilité de défaut
    et retourne la décision métier.
    """

    # Conversion en DataFrame
    X = pd.DataFrame([client_data])

    # Alignement des colonnes
    X = X.reindex(
        columns=feature_names,
        fill_value=0
    )

    # Probabilité de défaut
    default_probability = float(
        model.predict_proba(X)[0, 1]
    )

    # Décision métier
    decision = (
        "REFUSED"
        if default_probability >= threshold
        else "ACCEPTED"
    )

    return {
        "default_probability": round(
            default_probability,
            6
        ),
        "business_threshold": round(
            float(threshold),
            6
        ),
        "decision": decision
    }

In [28]:
# ==================================================
# TESTS DE LA FONCTION DE PREDICTION
# ==================================================

test_clients = {
    "CLIENT_STANDARD": {
        "AMT_INCOME_TOTAL": 200000,
        "AMT_CREDIT": 300000,
        "AMT_ANNUITY": 15000,
        "EXT_SOURCE_MEAN": 0.60,
        "PAYMENT_RATE": 0.05
    },

    "CLIENT_RISQUE": {
        "EXT_SOURCE_MEAN": 0.01,
        "EXT_SOURCE_2": 0.01,
        "EXT_SOURCE_3": 0.01,
        "PAYMENT_RATE": 0.01
    },

    "CLIENT_INCOMPLET": {}
}

for name, client in test_clients.items():

    print("=" * 50)
    print(name)
    print("=" * 50)

    result = predict_client(client)

    print(result)
    print()

CLIENT_STANDARD
{'default_probability': 0.450369, 'business_threshold': 0.090909, 'decision': 'REFUSED'}

CLIENT_RISQUE
{'default_probability': 0.755987, 'business_threshold': 0.090909, 'decision': 'REFUSED'}

CLIENT_INCOMPLET
{'default_probability': 0.409655, 'business_threshold': 0.090909, 'decision': 'REFUSED'}



## Validation de la règle métier

Le modèle produit une probabilité de défaut comprise entre 0 et 1.

La décision finale n'est pas prise directement à partir de cette probabilité mais à partir du seuil métier optimisé lors de la phase de modélisation.

Seuil retenu :

- seuil métier = 0.0909

Règle de décision :

- probabilité < seuil → crédit accepté (`ACCEPTED`)
- probabilité ≥ seuil → crédit refusé (`REFUSED`)

Les exemples ci-dessous permettent de vérifier que la règle métier est correctement appliquée.

In [30]:
# ==================================================
# TEST DE COHERENCE DE LA REGLE METIER
# ==================================================

print("SEUIL METIER :", round(float(threshold), 6))
print()

test_probabilities = [
    0.01,
    0.05,
    0.09,
    0.10,
    0.50
]

for p in test_probabilities:

    decision = (
        "REFUSED"
        if p >= threshold
        else "ACCEPTED"
    )

    print(
        f"Probabilité de défaut = {p:.2%} "
        f"| Seuil = {threshold:.2%} "
        f"| Décision = {decision}"
    )

SEUIL METIER : 0.090909

Probabilité de défaut = 1.00% | Seuil = 9.09% | Décision = ACCEPTED
Probabilité de défaut = 5.00% | Seuil = 9.09% | Décision = ACCEPTED
Probabilité de défaut = 9.00% | Seuil = 9.09% | Décision = ACCEPTED
Probabilité de défaut = 10.00% | Seuil = 9.09% | Décision = REFUSED
Probabilité de défaut = 50.00% | Seuil = 9.09% | Décision = REFUSED


### Interprétation

Les résultats montrent que le seuil métier est correctement appliqué :

- un client dont la probabilité de défaut est inférieure à 9.09 % est accepté ;
- un client dont la probabilité de défaut est supérieure ou égale à 9.09 % est refusé.

Cette règle sera utilisée dans l'API afin de transformer la probabilité prédite par le modèle en une décision métier exploitable par les équipes de crédit.

In [18]:
# ==================================================
# TEST ROBUSTESSE
# ==================================================

predict_client({})

{'default_probability': 0.409655,
 'business_threshold': 0.090909,
 'decision': 'REFUSED'}

In [20]:
# ==================================================
# VALIDATION DES ARTEFACTS
# ==================================================

print("Modèle chargé :", model.__class__.__name__)
print("Nombre de variables :", len(feature_names))
print("Seuil métier :", round(float(threshold), 6))

Modèle chargé : LGBMClassifier
Nombre de variables : 246
Seuil métier : 0.090909
